# BigQuery, introduced — and checked against what we've already built

**Home Safe (Kent domiciliary/personal care market intelligence) — LSA Data Engineering Internship, September 2026 cohort**
**Mentor note — Venkat, 10 Sept 2026**

This is a response to the BigQuery write-up that got shared in chat — worth doing properly rather than taking it at face value. Two things changed from the original:

1. **The example was rebuilt against our actual data, not generic ad metrics.** The original schema (`impressions`, `clicks`, `spend_usd`) is a real BigQuery pattern, but it's a marketing-analytics pattern — Home Safe's competitive question is regulatory and geographic (provider density and CQC rating by Kent district), not ad performance. Teaching material is more useful when it uses the columns you'll actually type.
2. **Every technical claim was checked against Google's own docs before being repeated here**, the same discipline as Weeks 1–2. One claim in the original — a percent-of-total query written as one bare nested expression — couldn't be confirmed either way; see Task 2 below for what's used instead and why.

**Where this fits:** the project spec's own Week 3 section says it plainly — *"SQLite is the baseline; a cloud warehouse is a stretch goal if time allows."* This notebook is what that stretch goal looks like. It is **not** a replacement for the SQLite plan, and the honest comparison in the next section explains why.

## What was actually checked before writing this, and how

**BigQuery Sandbox — confirmed via Google's own docs (10 Sept 2026):**
- No credit card or billing account required to start.
- Free limits: **10 GiB storage** (a *lifetime* cap, not monthly — deleting data doesn't refund it) and **1 TiB of query processing per month** (the same allowance as the standard BigQuery free tier).
- Tables, views, and partitions **auto-expire after 60 days**.
- **DML is disabled in the sandbox** — no `UPDATE`, `DELETE`, or `MERGE`. Streaming inserts and the Data Transfer Service are disabled too.
- Each person needs their own Google account / project — the free limits are per-project, not shared across the cohort.

**That DML restriction matters here, concretely:** the Week 3 spec's idempotent-load design is "delete-and-insert by month partition" — that's a DML pattern, and it will not run in sandbox mode as written. The workaround (below, Task 3) is a full-partition overwrite via a load job instead of a `DELETE`. Worth knowing before anyone builds the real Week 3 pipeline around it and hits this at the last minute.

**`SAFE_DIVIDE` — confirmed, used correctly in the original.** Returns `NULL` instead of erroring on divide-by-zero; exactly what it was used for.

**The percent-of-total pattern — only partly confirmed.** The original wrote it as one nested expression: `SUM(impressions) / SUM(SUM(impressions)) OVER()` inside a single `SELECT ... GROUP BY`. A working, docs-confirmed version of this pattern exists — but only when the grouped aggregate is computed first in a CTE or subquery, and the window function is applied in the query around it. Whether the fully-nested one-line form BigQuery also accepts it directly inside one `GROUP BY` query is genuinely unclear from the sources checked — two references disagreed. Rather than repeat a claim that couldn't be pinned down, this notebook uses the CTE form throughout (Task 2) — it's confirmed correct, and arguably clearer to read regardless.

**`bq load` from a local file — confirmed.** No Cloud Storage bucket needed for a dataset this size; a plain local-file load works (Task 4).

**Sources:** Google Cloud — [Try BigQuery using the sandbox](https://cloud.google.com/bigquery/docs/sandbox), [aggregate functions reference](https://cloud.google.com/bigquery/docs/reference/standard-sql/aggregate_functions), [bq command-line codelab](https://codelabs.developers.google.com/codelabs/bigquery-cli); percent-of-total pattern cross-checked against a third-party worked example (cited inline, Task 2) since the official reference didn't give a worked query.

## SQLite (the plan) vs. BigQuery Sandbox (the stretch goal) — the actual trade-off

| | SQLite (Week 3 baseline) | BigQuery Sandbox |
|---|---|---|
| **Setup** | Nothing — a file on disk | A Google account + project each; a few minutes |
| **Cost** | Free, no limits that matter here | Free within 10 GiB storage / 1 TiB query per month — plenty for Kent-sized data |
| **Idempotent load (spec's design)** | `DELETE` + `INSERT` works as written | **Doesn't** — DML is off; needs a load-job overwrite instead (Task 3) |
| **Data lifetime** | Permanent, yours | Auto-expires after 60 days unless reloaded |
| **Scale this project needs** | Comfortably enough — low hundreds of Kent rows | Massive headroom neither project nor sandbox will come close to using |
| **What it teaches** | Real relational modelling, SQL fundamentals | The same SQL, plus how a cloud warehouse actually behaves — partitioning, clustering, a genuinely different auth/quota model |
| **CV relevance** | Still the right foundation | Directly what a lot of data-analyst/data-engineer job specs name |

**Read on this:** for finishing the Week 3/4 deliverables on time, SQLite is still the right call — it's already the spec's baseline and nothing about Kent's data volume needs a warehouse. BigQuery Sandbox is worth doing as an **extension**, once the core pipeline works, for teams with time left over — mainly for the resume line and for seeing how the same modelling problem plays out on a real cloud platform. Not a Week 3 requirement swap.

## Environment check

In [ ]:
import sys

def check_importable(name, pip_name=None):
    try:
        __import__(name)
        print(f"OK  - {name}")
        return True
    except ImportError:
        print(f"MISSING - {name} (pip install {pip_name or name})")
        return False

print(f"Python: {sys.version}\n")
check_importable("pandas")
check_importable("google.cloud.bigquery", "google-cloud-bigquery")

# If MISSING:
# !pip install google-cloud-bigquery pandas --quiet

## Getting a sandbox project (do this once, before Week 3)

1. Go to `console.cloud.google.com/bigquery` and sign in with any Google account.
2. Create a project (or use the default one offered) — **no billing details requested**, that's the sandbox.
3. You land in the BigQuery Studio SQL editor immediately — no separate install needed for the SQL parts of this notebook.
4. For the Python client cells below (`google-cloud-bigquery`), authenticate locally with `gcloud auth application-default login`, or run this notebook in Colab and use `google.colab.auth.authenticate_user()`.

**Not yet verified:** the exact prompts on today's console sign-up flow — Google changes onboarding UI often enough that a step-by-step screenshot guide would go stale fast. Confirm live on your own screen against the two steps above rather than trusting a screenshot from someone else's run.

## Task 1 — Model Home Safe's actual data as two BigQuery tables

Reusing the real schema from Weeks 1–2, not inventing a new one:

- **`kent_locations`** — from Week 1's `home_safe_kent_personal_care_locations.csv`. One row per CQC *location* — this is where `overall_rating` lives.
- **`kent_providers_enriched`** — from Week 2's `home_safe_kent_providers_enriched.parquet`. One row per *provider* (the legal entity) — this is where the Companies House enrichment and `kent_area` live.

A provider can run several locations, so the real business question — density and rating by Kent area — needs a **join** between them, not just one table. That's a better BigQuery teaching example than a single flat table anyway: joins across differently-grained tables are exactly the kind of thing a warehouse is for.

In [ ]:
CREATE_LOCATIONS_TABLE = """
CREATE TABLE IF NOT EXISTS `home_safe.kent_locations` (
  location_id STRING,
  provider_id STRING,
  location_name STRING,
  postal_code STRING,
  local_authority STRING,
  registration_status STRING,
  overall_rating STRING,
  overall_rating_date DATE
)
"""

CREATE_PROVIDERS_TABLE = """
CREATE TABLE IF NOT EXISTS `home_safe.kent_providers_enriched` (
  provider_id STRING,
  provider_name STRING,
  kent_area STRING,
  companies_house_number STRING,
  match_method STRING,
  match_score FLOAT64,
  ch_company_status STRING,
  ch_date_of_creation DATE
)
"""

# Trimmed to the columns this notebook's queries actually use -- the real Week 2
# Parquet file has more (charity_number, postcode_valid, ...); add columns here
# to match it exactly, or let `bq load --autodetect` infer the full schema instead
# of hand-writing DDL at all (Task 4 uses autodetect for exactly this reason).
print(CREATE_LOCATIONS_TABLE)
print(CREATE_PROVIDERS_TABLE)

## Task 2 — The actual Week 4 business question, in SQL

*"Based on the number, type, and CQC quality rating of existing registered providers, where in Kent is domiciliary/personal care most saturated, and where is there a credible opening?"* — straight from the spec. Two queries: density + rating distribution by district, then market share by district using the confirmed-safe percent-of-total pattern.

In [ ]:
DENSITY_AND_RATING_QUERY = r'''
SELECT
  l.local_authority AS kent_area,
  COUNT(DISTINCT l.provider_id) AS provider_count,
  COUNT(DISTINCT l.location_id) AS location_count,
  COUNTIF(l.overall_rating = 'Outstanding') AS outstanding,
  COUNTIF(l.overall_rating = 'Good') AS good,
  COUNTIF(l.overall_rating = 'Requires improvement') AS requires_improvement,
  COUNTIF(l.overall_rating = 'Inadequate') AS inadequate,
  SAFE_DIVIDE(
    COUNTIF(l.overall_rating IN ('Outstanding', 'Good')),
    COUNT(l.overall_rating)
  ) AS good_or_better_rate
FROM `home_safe.kent_locations` AS l
GROUP BY kent_area
ORDER BY provider_count DESC
'''

print(DENSITY_AND_RATING_QUERY)

In [ ]:
# Market share by district -- CTE first (the confirmed-safe form), window
# function second. See "What was checked" above for why this shape, not the
# single-line nested form from the original write-up.
MARKET_SHARE_QUERY = r'''
WITH district_counts AS (
  SELECT
    local_authority AS kent_area,
    COUNT(DISTINCT provider_id) AS provider_count
  FROM `home_safe.kent_locations`
  GROUP BY kent_area
)
SELECT
  kent_area,
  provider_count,
  SAFE_DIVIDE(provider_count, SUM(provider_count) OVER ()) AS share_of_kent_providers
FROM district_counts
ORDER BY provider_count DESC
'''

print(MARKET_SHARE_QUERY)

## Task 3 — Bringing in the Companies House enrichment (a join, not a second flat table)

This is the query that actually needs BigQuery's join performance to matter at any real scale — cross-referencing rating with how well-established a provider's registered company is (`ch_date_of_creation`), and separately checking whether match quality varies by district (a data-quality question, not a market one, but worth watching).

In [ ]:
ENRICHED_VIEW_QUERY = r'''
SELECT
  l.local_authority AS kent_area,
  l.overall_rating,
  p.provider_name,
  p.match_method,
  p.match_score,
  p.ch_company_status,
  p.ch_date_of_creation
FROM `home_safe.kent_locations` AS l
JOIN `home_safe.kent_providers_enriched` AS p
  ON l.provider_id = p.provider_id
ORDER BY l.local_authority, l.overall_rating
'''

print(ENRICHED_VIEW_QUERY)

## Task 3b — The DML catch, and the workaround

The spec's Week 3 load design is idempotent via `DELETE` + `INSERT` by partition. **Sandbox mode blocks DML**, so that exact pattern will fail with a permissions/feature error if tried as written. The standard warehouse-native way around this: overwrite the whole table (or the relevant partition) with a **load job** set to `WRITE_TRUNCATE`, instead of deleting rows with SQL. It's a different idempotency mechanism — replace-the-whole-thing instead of delete-then-insert — but it achieves the same "re-running this safely doesn't duplicate rows" goal the spec is actually after.

In [ ]:
# Illustrative -- shows the *shape* of the fix, not a tested call (no live
# BigQuery credentials in this environment; see the note at the end of this
# notebook for what still needs running on your own machine).
from google.cloud import bigquery

def load_dataframe_idempotent(client, dataframe, table_id):
    """Overwrite `table_id` entirely with `dataframe`'s contents. This is the
    sandbox-compatible stand-in for the spec's DELETE+INSERT idempotent load --
    WRITE_TRUNCATE replaces the table's contents atomically, so re-running this
    twice with the same input never duplicates a row, same end result as the
    DML version, without needing DML at all."""
    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        autodetect=True,
    )
    job = client.load_table_from_dataframe(dataframe, table_id, job_config=job_config)
    job.result()  # blocks until the load finishes
    print(f"Loaded {len(dataframe)} rows into {table_id} (WRITE_TRUNCATE)")

print("Defined load_dataframe_idempotent -- see docstring for why this replaces DELETE+INSERT.")

## Task 4 — Actually loading Week 2's real output (once you have a project)

Two ways in, both confirmed to exist; pick whichever's more comfortable.

In [ ]:
# Command-line route (run this in a terminal, not this notebook's kernel --
# it needs `gcloud`/`bq` installed and authenticated):
BQ_LOAD_COMMAND = r'''
bq load \
  --source_format=PARQUET \
  --autodetect \
  home_safe.kent_providers_enriched \
  ./home_safe_kent_providers_enriched.parquet
'''
print(BQ_LOAD_COMMAND)
print("No Cloud Storage bucket needed for a file this size -- bq load takes a local path directly.")

In [ ]:
# Python client route -- same load, from inside a notebook. Illustrative: this
# cell has not been run against live BigQuery in this environment (no
# credentials here), only checked for syntax and against the client library's
# own documented API.
import pandas as pd
from google.cloud import bigquery

def load_week2_output_to_bigquery(parquet_path, table_id):
    client = bigquery.Client()  # picks up application-default credentials
    df = pd.read_parquet(parquet_path)
    job_config = bigquery.LoadJobConfig(autodetect=True, write_disposition="WRITE_TRUNCATE")
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()
    print(f"Loaded {len(df)} rows into {table_id}")

# load_week2_output_to_bigquery(
#     "../Week_2/home_safe_kent_providers_enriched.parquet",
#     "your-project.home_safe.kent_providers_enriched",
# )
print("google-cloud-bigquery import check:")
print(bigquery.__name__, "available -- client library import confirmed in this environment.")

## What's confirmed vs. what still needs a live run

**Confirmed by checking Google's own documentation (10 Sept 2026):** sandbox has no billing requirement, the specific free-tier limits quoted above, DML being disabled in sandbox, `SAFE_DIVIDE`'s behaviour, the CTE-based percent-of-total pattern, and `bq load` accepting a local file path.

**Not yet confirmed — needs someone to actually run it:**
- Every query above, against a real loaded table. Syntax was checked carefully; none of it has been executed against live BigQuery, because this environment has no GCP credentials and (same as CQC and Companies House in Weeks 1–2) its own network is proxy-restricted to a small allow-list. Same rule as always: this is confirmed-checked, not confirmed-executed. Run it, don't just trust it.
- Whether the field names in `CREATE TABLE` above exactly match what `bq load --autodetect` infers from the real Week 1/2 files — autodetect is usually reliable but worth a `bq show` afterwards to confirm column types, especially for the date fields.
- Whether the bare single-line nested percent-of-total form works after all — genuinely unresolved here; if someone tries it and it works, that's worth reporting back, not just assuming the CTE form was the only option.

**On the original write-up's closing offer** — a Looker Studio dashboard on top of this: worth having in mind for later, but Week 4's dashboard deliverable is already planned against the Python/pandas stack from Weeks 1–2. Doubling that up is a decision for whoever's actually building Week 4, not something to build ahead of being asked for.

**Sources:** [BigQuery sandbox](https://cloud.google.com/bigquery/docs/sandbox); [BigQuery aggregate functions reference](https://cloud.google.com/bigquery/docs/reference/standard-sql/aggregate_functions); [bq command-line tool codelab](https://codelabs.developers.google.com/codelabs/bigquery-cli); `HomeSafe_Week1-4_Project_Spec.md`, Week 3 section; Week 1/2 notebooks (`home_safe_kent_personal_care_locations.csv`, `home_safe_kent_providers_enriched.parquet` schemas).